<a target="_blank" href="https://colab.research.google.com/github/seap-udea/fargopy/blob/main/examples/tutorials/planets-turorial.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

<p align="left"><img src="https://github.com/seap-udea/fargopy/raw/refactor/docs/fargopy_logo.webp" width="300" /></p>

# Planet data tutorial

This tutorial shows how to load planet data from a FARGO3D/FARGOpy simulation, convert the quantities from code units to physical units, and reconstruct the planet trajectory across snapshots.

The examples are written for the precomputed `p3diso` simulation, but the same workflow applies to any FARGO3D output directory loaded with `fp.Simulation(output_dir=...)`.

## 1. Reference frames in FARGO3D

Before analyzing the planet position, check the reference frame used by the simulation. In FARGO3D, the parameter `Frame` controls how the computational frame rotates:

- `Frame = F` or `Fixed`: the frame rotates at a constant angular velocity set by `OmegaFrame`.
- `Frame = C` or `Corotating`: the frame corotates with planet 0. If that planet migrates or has an eccentric orbit, the frame angular velocity is time-dependent.
- `Frame = G` or `Guiding-Center`: the frame corotates with the guiding center of planet 0, giving a smoother time-dependent frame when the planet migrates.

For diagnostics centered on a planet, such as circumplanetary-disk structure, Hill-sphere cuts, accretion diagnostics, or streamlines around the planet, a corotating frame is often convenient because the planet can remain approximately fixed in the grid. In that case, the orbit plot in the rotating frame may collapse to one point. This is not an error; it means the selected planet is stationary in the chosen computational frame.

For global orbital evolution, migration, or eccentric motion, use the snapshot-by-snapshot planet positions and interpret them consistently with the frame stored in the output files.

In [1]:
import fargopy as fp
import numpy as np
import matplotlib.pyplot as plt

try:
    import ipywidgets as widgets
    from IPython.display import display
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False

%load_ext autoreload
%autoreload 2

Running FARGOpy version 1.2.0.
NOTE: Since alpha versions (<=0.X.X) a major refactor has been done in versions 1.1.X.
Please check the documentation for more information.


## 2. Load a precomputed simulation

Here we use the precomputed `p3diso` simulation. You can replace this block with your own output directory:

```python
sim = fp.Simulation(output_dir="/path/to/your/output")
```

In [2]:
dir = fp.Simulation.download_precomputed("p3diso")
sim = fp.Simulation(output_dir=dir)

Downloading...
From (original): https://drive.google.com/uc?id=1KMp_82ylQn3ne_aNWEF1T9ElX2aWzYX6
From (redirected): https://drive.google.com/uc?id=1KMp_82ylQn3ne_aNWEF1T9ElX2aWzYX6&confirm=t&uuid=11845d60-a30a-47ce-bbce-3d3d7d9e4a02
To: /tmp/p3diso.tgz
100%|██████████| 230M/230M [00:17<00:00, 13.0MB/s] 


Uncompressing p3diso.tgz into /tmp/p3diso
Done.
Your simulation is now connected with '/home/fargopy/fargo3d/'
Now you are connected with output directory '/tmp/p3diso'
Found a variables.par file in '/tmp/p3diso', loading properties
Loading variables
85 variables loaded
Simulation in 3 dimensions
Loading domain in spherical coordinates:
	Variable phi: 100 [[0, np.float64(-3.1101767270538954)], [-1, np.float64(3.110176727053896)]]
	Variable r: 80 [[0, np.float64(0.605625)], [-1, np.float64(1.494375)]]
	Variable theta: 20 [[0, np.float64(1.4245463267948968)], [-1, np.float64(1.5670463267948964)]]
Number of snapshots in output directory: 50
Planets found in summary.dat
  Name: SuperEarth, Initial pos: [1.0, 1e-05, 0.0], Mass: 1e-05


## 3. Define physical units

If the simulation was compiled with `UNITS=0 RESCALE=0`, FARGO3D stores the output in code units. You can define the conversion from code units to physical units with `sim.set_units`.

For example, if:

- `R0 = 1.0` corresponds to 5.8 AU
- `MSTAR = 1.0` corresponds to 1 solar mass

then use:

```python
sim.set_units(UL=5.8 * fp.AU, UM=fp.MSUN)
```

The relevant conversion factors are:

- `sim.UL`: length unit
- `sim.UM`: mass unit
- `sim.UT`: time unit
- `sim.UV`: velocity unit
- `sim.URHO`: volume-density unit
- `sim.USIGMA`: surface-density unit

In [10]:
# Edit these values for your simulation.
R0_AU = 5.8
MSTAR_MSUN = 1.0

sim.set_units(
    UL=R0_AU * sim.AU,
    UM=MSTAR_MSUN * sim.MSUN,
)

MJUP = 1.89813e30  # g
MEARTH = 5.9722e27 # g

print("Unit conversions")
print(f"UL = {sim.UL:.6e} cm = {sim.UL / sim.AU:.6f} AU")
print(f"UM = {sim.UM:.6e} g = {sim.UM / sim.MSUN:.6f} Msun")
print(f"UT = {sim.UT:.6e} s = {sim.UT / sim.YEAR:.6f} yr")
print(f"UV = {sim.UV:.6e} cm/s = {sim.UV / 1e5:.6f} km/s")

Unit conversions
UL = 8.676684e+13 cm = 5.800000 AU
UM = 1.989100e+33 g = 1.000000 Msun
UT = 7.015444e+07 s = 2.223060 yr
UV = 1.236798e+06 cm/s = 12.367976 km/s


## 4. Load planet data at one snapshot

`sim.load_planets(snapshot=n)` reads the planet section of the corresponding `summary<n>.dat` file and returns a list of `Planet` objects. Each planet stores position, velocity, mass and Hill radius.

In [4]:
snapshot = 10
planets = sim.load_planets(snapshot=snapshot)

print(f"Number of planets at snapshot {snapshot}: {len(planets)}")
for i, p in enumerate(planets):
    print(f"{i}: name={p.name}, mass_code={p.mass:.6e}, pos_code=({p.pos.x:.6e}, {p.pos.y:.6e}, {p.pos.z:.6e})")

Number of planets at snapshot 10: 1
0: name=SuperEarth, mass_code=1.000000e-05, pos_code=(9.999999e-01, -4.712471e-04, 0.000000e+00)


## 5. Convert one planet to physical units

Positions are converted with `sim.UL`. Masses are converted with `sim.UM`. The Hill radius returned by FARGOpy is in the same length unit as the planet position, so it is converted in the same way.

In [5]:
planet_index = 0
p = planets[planet_index]

x_code, y_code, z_code = p.pos.x, p.pos.y, p.pos.z
vx_code, vy_code, vz_code = p.vel.x, p.vel.y, p.vel.z
m_code = p.mass
rhill_code = p.hill_radius

x_AU = x_code * sim.UL / fp.AU
y_AU = y_code * sim.UL / fp.AU
z_AU = z_code * sim.UL / fp.AU

vx_kms = vx_code * sim.UV / 1e5
vy_kms = vy_code * sim.UV / 1e5
vz_kms = vz_code * sim.UV / 1e5

m_Msun = m_code * sim.UM / fp.MSUN
m_Mjup = m_code * sim.UM / MJUP
m_Mearth = m_code * sim.UM / MEARTH

rhill_AU = rhill_code * sim.UL / fp.AU

print(f"Planet {planet_index}: {p.name}")
print(f"Position [code] = ({x_code:.6e}, {y_code:.6e}, {z_code:.6e})")
print(f"Position [AU]   = ({x_AU:.6e}, {y_AU:.6e}, {z_AU:.6e})")
print(f"Velocity [km/s] = ({vx_kms:.6e}, {vy_kms:.6e}, {vz_kms:.6e})")
print(f"Mass = {m_Msun:.6e} Msun = {m_Mjup:.6e} Mjup = {m_Mearth:.6e} Mearth")
print(f"Hill radius = {rhill_code:.6e} code units = {rhill_AU:.6e} AU")

Planet 0: SuperEarth
Position [code] = (9.999999e-01, -4.712471e-04, 0.000000e+00)
Position [AU]   = (5.799999e+00, -2.733233e-03, 0.000000e+00)
Velocity [km/s] = (5.828344e-03, 1.236804e+01, 0.000000e+00)
Mass = 1.000000e-05 Msun = 1.047926e-02 Mjup = 3.330598e+00 Mearth
Hill radius = 1.470664e-02 code units = 8.529850e-02 AU


## 6. Practical notes

Use a corotating frame when the diagnostic is local to the planet, for example CPD morphology, Hill-sphere cuts, gas capture, or circumplanetary streamlines. In this case a fixed planet position is desirable.

Use the full trajectory across snapshots when studying migration, eccentricity, mass growth, or the motion of a non-fixed planet. If the planet appears fixed, inspect the FARGO3D `Frame` parameter before interpreting the orbit as physical motion in an inertial frame.

Always apply the same unit conversion consistently to the gas fields, planet position, Hill radius and any derived diagnostic.

---
*Powered by fargopy*. For more examples see [fargopy GitHub repo](https://github.com/seap-udea/fargopy/tree/main/examples). 

Jorge I. Zuluaga, Alejandro Murillo-González and Matías Montesinos © 2023-present
